# Release notes
Paths and language were normalized for this source release. Historical training protocols remain documented in `REVIEW-1023.md`; retrain after any protocol corrections.


In [ ]:
from pathlib import Path
import sys
import copy
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "models/DeepLearnModel.py").is_file() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the MMHRP-CAMoPE-1023 project.")
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install rdkit==2024.9.5
# !pip install torch_geometric==2.5.3

In [ ]:
import os, sys

# Inspect the current working directory
print("Current working directory:", os.getcwd())

# Inspect the Python module search path
print("Python module search path:")
for p in sys.path:
    print(p)


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

import sys
import torch
from torch.utils.data import DataLoader
# doc_name = str(PROJECT_ROOT / '')
doc_name = str(PROJECT_ROOT)
sys.path.append(doc_name)
from utils.rxn import *
from utils.molecule import *
from torch_geometric.loader import DataLoader
from models.DeepLearnModel import *
import time
from tqdm import tqdm
import datetime
import warnings
warnings.simplefilter('ignore')

In [ ]:
# rs_list = [1,2,3,4,5]
rs_list = [1]
eval_metrics = np.zeros((1+len(rs_list), 6))
columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
index = []
for rs in rs_list:
  index.append("%s" % rs)
index.append("avg±std")
eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)



In [ ]:
for rs in rs_list:
    # 1. import data
    data = pd.read_excel("%s/data/Heck/Heck_data_clean.xlsx" % doc_name)
    random_state = rs

    data = data.sample(random_state=random_state, frac=1).reset_index(drop=True)
    vocab_path = "%s/models/AHO_vocab.txt" % doc_name

    # 2. build dataset & dataloader
    rxn_list = df_to_rxn_list(data)
    rxn_RxnSmi = list()
    max_len = -1
    for rxn in rxn_list:
      RxnSmi = get_Heck_RxnSmi(rxn)
      max_len = max(max_len, len(RxnSmi))
      RxnSmi = " ".join(smi_tokenizer(RxnSmi))
      rxn_RxnSmi.append(RxnSmi)

    rxn_dataset = list()
    smi_inputsize = 128

    for batch in tqdm(range(len(rxn_list))):
        rxn = rxn_list[batch]
        meta = list()
        # reactants & products
        reac = rxn.reactants
        prod = rxn.products
        meta.append(smis_to_graph(reac + prod, ChargeContribution=False))
        # reagents
        sols = rxn.solvents
        cats = rxn.cats
        reag = rxn.reagents
        meta.append(smis_to_graph(sols + cats + reag, ChargeContribution=False))

        # RxnSmi
        RxnSmi_vec = RxnSmi_to_tensor(RxnSmi=rxn_RxnSmi[batch], maxlen_=max_len, victor_size=smi_inputsize,
                                      file=vocab_path)
        meta.append(RxnSmi_vec)
        
        # Condition: temperature & time
        cond_feat = torch.tensor([rxn.temp, rxn.time], dtype=torch.float32)
        meta.append(cond_feat)

        # yield
        meta.append(rxn.rxn_yield / 100)

        rxn_dataset.append(meta)

    # report
    # dir_path = "/exp/Heck_Benchmark/Heck_MMHRP-GCL_rs=%s" % (random_state)
    # os.mkdir("%s" % dir_path)
    # f = open("%s/Model_Training_Report.txt" % dir_path, mode="w")

    # base_dir = str(PROJECT_ROOT / 'exp/Heck_Benchmark')
    base_dir = f"{doc_name}/exp/Heck_Benchmark/main"
    
    # Construct the complete path
    dir_path = os.path.join(base_dir, f"MMHRP_GCL_tri_MoPE_pdall_IR_64_rs=2")
    
    
    
    # dir_path = str(PROJECT_ROOT / 'exp/Heck_Benchmark/Heck_MMHRP-GCL-tri_rs=%s') % (random_state)
    # os.mkdir("%s" % dir_path)
    # f = open("%s/Model_Training_Report.txt" % dir_path, mode="w")
      
    
    # Create the directory and any missing parents
    os.makedirs(dir_path, exist_ok=True)
    
    # Open the report file
    report_path = os.path.join(dir_path, "Model_Training_Report.txt")
    f = open(report_path, mode="w")

    # split of train & test set
    ratio = 0.8
    batch_size = 64
    batch = len(rxn_dataset)
    train_set = rxn_dataset[0: int(ratio * batch)]
    test_set = rxn_dataset[int(ratio * batch) + 1:]
    # data_loader
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=20, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=20, pin_memory=True, persistent_workers=True)

    test_RMSE = list()
    test_R2 = list()
    test_MAE = list()
    train_R2 = list()
    train_RMSE = list()
    train_MAE = list()
    pred = list()
    true = list()

    # 3. training of the model
    # params
    t = 900  # Original setting: 900 epochs
    lr = 1e-4
    initial_lr = lr
    best_model_path = os.path.join(dir_path, "best_model.pth")
    
    num_feature = 7
    smi_inputsize = 128
    Cond_OutDim = 32

    # use gpu
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # device = torch.device("cpu")

    # model
    model = MMHRP_GCL_tri_MoPE_pdall(
        GraphEncoder={
            "NodeFeatNum": num_feature,
            "Channels": [16, 32],
            "Heads": 4
        },
        TextEncoder={
            "SmiFeatNum": smi_inputsize,
            "Heads": 2,
            "BigruChannels": [128, 128],
            "BigruNumlayer": 1
        },
        ConditionEncoder={
            "InputDim": 2,  # Temperature and reaction time
            "HiddenDim": 64,
            "OutDim": Cond_OutDim,
        },
        Decoder={
            "Channels": [128, 64, 32]
        },
        emb_size=128,
        device=device,
        FDS=None  # Optional feature distribution smoothing; disabled here
    )
    model = model.to(device)
    opti = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()
    
    scheduler = torch.optim.lr_scheduler.StepLR(opti, step_size=5, gamma=0.1)

    # writedown params
    f.write("params:\n")
    f.write("random_state=%s\n" % random_state)
    f.write("ratio=%s\n" % ratio)
    f.write("batch_size=%s\n" % batch_size)
    f.write("t=%s\n" % t)
    f.write("lr=%s\n" % lr)

    # Training

    # best performance
    best = [0, 0, 0, 0, 0, 0, [], [],
            []]  # train_R2, train_RMSE, train_MAE, test_R2, test_RMSE, test_MAE test_predict, test_true, model

    f.write("\nStart training\n")

    for epoch in tqdm(range(t)):
        # Training
        global_loss = torch.tensor([0.])
        current_lr = opti.param_groups[0]["lr"]

        for data in train_loader:
            x = [i.to(device) for i in data[:-1]]
            y = torch.unsqueeze(data[-1], dim=1).to(device)
            loss = criterion(model.forward(x), y)
            opti.zero_grad()
            loss.backward()
            opti.step()
            global_loss += loss.item()
            
        scheduler.step()

        # record of loss during training
        # performance in train set
        with torch.no_grad():
            pred = list()
            true = list()
            for data in train_loader:
                x = [i.to(device) for i in data[:-1]]
                tr = list(torch.unsqueeze(data[-1], dim=1).detach().numpy())
                pr = list(model.forward(x).cpu().detach().numpy())
                pred += pr
                true += tr
            train_RMSE.append(RMSE(np.array(pred), np.array(true)))
            train_R2.append(R2(np.array(pred), np.array(true)))
            train_MAE.append(MAE(np.array(true), np.array(pred)))

        # performance in test set
        with torch.no_grad():
            pred = list()
            true = list()
            for data in test_loader:
                x = [i.to(device) for i in data[:-1]]
                tr = list(torch.unsqueeze(data[-1], dim=1).detach().numpy())
                pr = list(model.forward(x).cpu().detach().numpy())
                pred += pr
                true += tr
            test_RMSE.append(RMSE(np.array(pred), np.array(true)))
            test_R2.append(R2(np.array(pred), np.array(true)))
            test_MAE.append(MAE(np.array(true), np.array(pred)))

            if epoch == 0 or test_R2[-1] > best[3]:
                best = [train_R2[-1], train_RMSE[-1], train_MAE[-1], test_R2[-1], test_RMSE[-1], test_MAE[-1], pred,
                        true, copy.deepcopy(model.state_dict()), current_lr]

        # write report
        f.write(
            "Epoch:%d lr: %.6e loss: %f, R2:train set %.3f\ttest set %.3f\tRMSE:train set %.3f\ttest set %.3f\tMAE:train set %.3f\ttest set %.3f\n" % (
            epoch + 1, current_lr, global_loss / batch_size, train_R2[-1], test_R2[-1], train_RMSE[-1], test_RMSE[-1],
            train_MAE[-1], test_MAE[-1]))
        
        if (epoch + 1) % 30 == 0:
            for pg in opti.param_groups:
                pg["lr"] = initial_lr

            scheduler = torch.optim.lr_scheduler.StepLR(
                opti, step_size=5, gamma=0.1
            )

            if os.path.exists(best_model_path):
                model.load_state_dict(torch.load(best_model_path))
                print(f"🔥 Warm restart at epoch {epoch+1}")

    # 4.Evaluation
    f.write("\n")
    # Performance in train set
    f.write("R2 of train set is:%.3f+-%f\tbest:%f\n" % (
    np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
    f.write("RMSE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))
    f.write("MAE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_MAE[-10:]).mean(), np.array(train_MAE[-10:]).std(), best[2]))

    # Performance in test set
    f.write("R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (
    np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
    f.write("RMSE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))
    f.write("MAE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_MAE[-10:]).mean(), np.array(test_MAE[-10:]).std(), best[5]))
    
    f.write("Best model learning rate: %.6e\n" % best[-1])
    print("Best model learning rate: %.6e\n" % best[-1])


    f.close()

    # Performance in train set
    print("R2 of train set is:%.3f+-%f\tbest:%f\n" % (
    np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
    print("RMSE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))
    print("MAE of train set is: %.3f+-%f\tbest:%f\n" % (
    np.array(train_MAE[-10:]).mean(), np.array(train_MAE[-10:]).std(), best[2]))

    # Performance in test set
    print("R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (
    np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
    print("RMSE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))
    print("MAE of test set is: %.3f+-%f\tbest:%f\n" % (
    np.array(test_MAE[-10:]).mean(), np.array(test_MAE[-10:]).std(), best[5]))

    eval_metrics.loc["%s" % rs]["train_R2"] = best[0]
    eval_metrics.loc["%s" % rs]["train_RMSE"] = best[1]
    eval_metrics.loc["%s" % rs]["train_MAE"] = best[2]
    eval_metrics.loc["%s" % rs]["test_R2"] = best[3]
    eval_metrics.loc["%s" % rs]["test_RMSE"] = best[4]
    eval_metrics.loc["%s" % rs]["test_MAE"] = best[5]

    # 5.Figure
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig = plt.figure(dpi=300, figsize=(10, 5))

    # Training Fig
    plt.subplot(1, 2, 1)
    steps = np.linspace(1, t, t)
    plt.plot(steps, train_R2, color=[236 / 255, 164 / 255, 124 / 255])
    plt.plot(steps, test_R2, color=[117 / 255, 157 / 255, 219 / 255])
    # Beautify
    plt.legend(["train set", "test set"], loc="upper left", prop={'size': 8})
    plt.xlabel("Epoch", fontsize=10)
    plt.ylabel("R$^2$", fontsize=10)
    plt.title("The R$^2$ of train & test set during training", fontsize=13)

    # Test set performance
    plt.subplot(1, 2, 2)
    tr = np.array(best[7]).flatten() * 100
    pr = np.array(best[6]).flatten() * 100
    plt.scatter(pr, tr, alpha=0.7, marker=".")
    plt.xlabel("Predicted Yield (%)", fontsize=10)
    plt.ylabel("Observed Yield (%)", fontsize=10)
    x = np.linspace(0, 100, 100)
    y = np.linspace(0, 100, 100)
    plt.plot(x, y, linestyle="--", color="r")
    plt.title("Test set performance", fontsize=15)

    fig.suptitle("MMHRP_GCL_tri_MoPE_pdall for HeckLit dataset", fontsize=16)
    plt.tight_layout()
    plt.savefig("%s/Performance_Figure.png" % dir_path)
    plt.show()

    torch.save(best[8], "%s/model.pth" % dir_path)

In [ ]:
# # Evaluation metrics report
# for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
#   for j in range(eval_metrics.shape[1]):
#     eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
# eval_metrics.to_csv("%s/exp/Heck_Benchmark/MMHRP-GCL_report_%s.csv" % (doc_name, datetime.datetime.now()))
# print(eval_metrics)

In [ ]:
# Evaluation metrics report
for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
  for j in range(eval_metrics.shape[1]):
    eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
# eval_metrics.to_csv("%s/exp/Heck_Benchmark/MMHRP-GCL_report_%s.csv" % (doc_name, datetime.datetime.now()))
eval_metrics.to_csv(str(PROJECT_ROOT / 'exp/Heck_Benchmark/main/MMHRP_GCL_tri_MoPE_pdall_IR_64_rs=2/MMHRP_GCL_tri_MoPE_pdall_IR_64_rs=2.csv') )#% (doc_name, datetime.datetime.now()
print(eval_metrics)